# GraphRAG vs. Baseline RAG: Solving Multi-Hop Reasoning

This notebook builds a **GraphRAG** index and a traditional **vector RAG** index over the
same corpus of articles, then compares them on multi-hop reasoning questions using
[LlamaIndex](https://www.llamaindex.ai/).

See the accompanying `README.md` for setup and prerequisites.

---

*Written and developed by **Amin Amiri**. Released under the MIT License.*


## Initial setup and utility function definitions


In [ ]:
import nest_asyncio
from dotenv import load_dotenv

load_dotenv("./.env")
nest_asyncio.apply()

def pretty_print_response(response):
    # Extract main components
    print("Response:")
    print("---------")
    print(f"Main response text: {response.response}\n")
    
    print("Source Nodes:")
    print("-------------")
    for i, node in enumerate(response.source_nodes, 1):
        print(f"\nNode {i}:")
        print(f"ID: {node.node.id_}")
        print(f"Text snippet: {node.node.text}")
        print(f"Score: {node.score}")
        print("-" * 50)

## Parse and load corpus of articles

In [ ]:
from llama_index.readers.json import JSONReader

reader = JSONReader()

documents = reader.load_data(input_file="./data/corpus_complete.json")
# documents = reader.load_data(input_file="./data/corpus_subset.json")

## GraphRAG index construction

In [ ]:
from llama_index.core import PropertyGraphIndex, StorageContext, load_index_from_storage
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
import os

GRAPH_RAG_PERSIST_DIR = "./storage/graph_rag"
# GRAPH_RAG_PERSIST_DIR = "./storage/graph_rag_subset"

if not os.path.exists(GRAPH_RAG_PERSIST_DIR):
    os.makedirs(GRAPH_RAG_PERSIST_DIR, exist_ok=True)
    index = PropertyGraphIndex.from_documents(
        documents,
        llm=OpenAI(model="gpt-4o-mini"),
        embed_model=OpenAIEmbedding(model_name="text-embedding-3-small"),
        show_progress=True,
    )
    index.storage_context.persist(persist_dir=GRAPH_RAG_PERSIST_DIR)
else:
    index = load_index_from_storage(
        StorageContext.from_defaults(persist_dir=GRAPH_RAG_PERSIST_DIR)
    )

query_engine_KnowledgeGraph = index.as_query_engine(
    include_text=True,
)

### Visualize the created knowledge graph

In [ ]:
index.property_graph_store.save_networkx_graph(name="./knowledge_graph.html")

Open the new file `knowledge_graph.html` in your browser to visualize the graph.

## Traditional Vector RAG index construction

In [ ]:
from llama_index.core import VectorStoreIndex

VECTOR_RAG_PERSIST_DIR = "./storage/vector_rag"

if not os.path.exists(VECTOR_RAG_PERSIST_DIR):
    index = VectorStoreIndex.from_documents(documents, show_progress=True)
    index.storage_context.persist(persist_dir=VECTOR_RAG_PERSIST_DIR)
else:
    index = load_index_from_storage(
        StorageContext.from_defaults(persist_dir=VECTOR_RAG_PERSIST_DIR)
)

query_engine_vector_RAG = index.as_query_engine()

# Comparing GraphRAG vs Vector RAG Performance

## **Question 1:** Who became a prominent figure in generative AI technology, notably with ChatGPT, and was recently the subject of controversy involving his departure from OpenAI, as discussed in both Fortune and TechCrunch articles?

- **Ground truth answer:**  
- **Question type:** Inference Query


### GraphRAG Query

In [ ]:
response = query_engine_KnowledgeGraph.query("Who became a prominent figure in generative AI technology, notably with ChatGPT, and was recently the subject of controversy involving his departure from OpenAI, as discussed in both Fortune and TechCrunch articles? Short answer.")
pretty_print_response(response)

### Vector RAG Query

In [ ]:
response = query_engine_vector_RAG.query("Who became a prominent figure in generative AI technology, notably with ChatGPT, and was recently the subject of controversy involving his departure from OpenAI, as discussed in both Fortune and TechCrunch articles? Short answer.")
pretty_print_response(response)

## **Question 2:** Does 'The Verge' article suggest that Sam Bankman-Fried set withdrawal permissions based on FTX's total trading revenue, while 'Fortune' and 'TechCrunch' articles focus on the jury's determination of his truthfulness and allegations of committing fraud for personal gain, respectively, without mentioning specific operational practices like withdrawal permissions?

- **Ground truth answer:** Yes
- **Question type:** Comparison Query

In [ ]:
response = query_engine_KnowledgeGraph.query("Does 'The Verge' article suggest that Sam Bankman-Fried set withdrawal permissions based on FTX's total trading revenue, while 'Fortune' and 'TechCrunch' articles focus on the jury's determination of his truthfulness and allegations of committing fraud for personal gain, respectively, without mentioning specific operational practices like withdrawal permissions?")
pretty_print_response(response)

In [ ]:
response = query_engine_vector_RAG.query("Does 'The Verge' article suggest that Sam Bankman-Fried set withdrawal permissions based on FTX's total trading revenue, while 'Fortune' and 'TechCrunch' articles focus on the jury's determination of his truthfulness and allegations of committing fraud for personal gain, respectively, without mentioning specific operational practices like withdrawal permissions?")
pretty_print_response(response)

## **Question 3:** Which entity is currently engaged with Amazon to address competition concerns, facilitating dialogue with consumer groups against Meta, deploying staff within its AI Office for future regulations, and has previously focused on illegal content and disinformation issues related to the Israel-Hamas war, as reported by TechCrunch?

- **Ground truth answer:** The European Commission
- **Question type:** Inference Query

### GraphRAG Query

In [ ]:
response = query_engine_KnowledgeGraph.query("Which entity is currently engaged with Amazon to address competition concerns, facilitating dialogue with consumer groups against Meta, deploying staff within its AI Office for future regulations, and has previously focused on illegal content and disinformation issues related to the Israel-Hamas war, as reported by TechCrunch? Short answer.")
pretty_print_response(response)

### Vector RAG Query

In [ ]:
response = query_engine_vector_RAG.query("Which entity is currently engaged with Amazon to address competition concerns, facilitating dialogue with consumer groups against Meta, deploying staff within its AI Office for future regulations, and has previously focused on illegal content and disinformation issues related to the Israel-Hamas war, as reported by TechCrunch? Short answer.")
pretty_print_response(response)

## Results

GraphRAG
| Question | Answer | Ground Truth | Correct? |
| -------- | -------- | ----------- | -------- |
| 1        | Sam Altman | Sam Altman | Yes      |
| 2        | Yes      | Yes         | Yes      |
| 3        | The European Commission | The European Commission | Yes      |

Vector RAG
| Question | Answer | Ground Truth | Correct? |
| -------- | -------- | ----------- | -------- |
| 1        | Dave Willner | Sam Altman | No      |
| 2        | No      | Yes         | No      |
| 3        | Microsoft | The European Commission | No      |

